# Threes model retrain: 2026-27

Spec chosen on a 2025-26 holdout (train 2022-23 to 2024-25), October 2026:
- All 12 incumbent features, outcome classes 0-9+ (`yMax = 9`).
- `cornerRatio` binned with `MulticlassOptimalBinning` fit against the outcome capped at 7, replacing the binary `OptimalBinning`.
- Rows filtered to `eligible == 1` **after** `threes_pipe`, so the rolling features keep each player's full game history.
- Rows with no minutes are dropped. They are placeholders for box scores that were never loaded (all of 2026-04-04 and 2026-04-12, plus scattered 2024-25 dates), not real zero-make games.

Holdout results (multiclass Brier collapsed to 0-7+, lower is better): production 0.71311, refit with the fixed features 0.71260, plus multiclass bins 0.71248. Over/under Brier is flat across all three.

Rejected: capping at 7+ (no gain, and it can't price Over 7.5 and above); dropping `crn_fgaMv`/`abv_fgaMv` (over/under Brier +0.0007, significant).

**Known issue:** on the holdout the refit over-predicts the tails more than production did: mean P(over 5.5) is 21% above observed, against 11% for production. The multiclass bins are not the cause; the plain refit shows the same. The final fit includes 2025-26, but check over-line calibration early in the season and be wary of Over 4.5+ edges until then.

In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import statsmodels.api as sm
from optbinning import MulticlassOptimalBinning
from nba import NBAdata, NBAmodels
from nba.constants import _BASE_DIR

data = NBAdata.data()
threes = NBAmodels.models('threes')

(CVXPY) Oct 09 01:05:03 PM: Encountered unexpected exception importing solver HIGHS:
ImportError('/home/claude/minienv/lib/python3.11/site-packages/highspy/_core.cpython-311-x86_64-linux-gnu.so: undefined symbol: _ZN5Highs13releaseMemoryEv')


In [2]:
FEATS = ['cornerRatio', 'abv_interaction', 'crn_interaction', 'mvAvgThrees', 'daysBetweenGames', 'wide_interaction',
         'home', 'crn_fgaMv', 'abv_fgaMv', 'threes_residualsAllowedMv', 'past3ThrPtPrct', 'mvGood3Rate']
NOSCALE = ['cornerRatio', 'daysBetweenGames', 'home', 'threes_residualsAllowedMv']
Y_MAX = 9
BIN_CAP = 7
TRAIN_START, HOLDOUT_START = '2022-10-01', '2025-10-01'
RUN_DIR = os.path.join(_BASE_DIR, 'data', 'model', '2026-27Run')

## Build the training frame

In [3]:
td = data.clean_na(data.threes_pipe(threes.data))
td = td[(td.eligible == 1) & (td.game_date >= TRAIN_START)].dropna(subset=FEATS + ['threesMade']).copy()
# rows for games whose box score was never loaded have NULL minutes and a placeholder 0 threes; they are not outcomes
mins = pd.read_sql('SELECT player_id, game_date, min FROM pgames', data.conn)
td = td.merge(mins, on=['player_id', 'game_date'], how='left')
print('dropping {:,} rows with no minutes'.format((td['min'].fillna(0) <= 0).sum()))
td = td[td['min'] > 0].copy()
td['rawRatio'] = td.abv_fgaMv / (td.abv_fgaMv + td.crn_fgaMv)
td['y'] = td.threesMade.clip(upper=Y_MAX).astype(int)
print('{:,} rows'.format(len(td)))
td.groupby('season').size()

dropping 464 rows with no minutes
72,886 rows


season
2022-23    17195
2023-24    18409
2024-25    18454
2025-26    18828
dtype: int64

## Holdout check: train through 2024-25, score 2025-26
Reproduces the comparison so the spec is checked on this machine's data before the final fit.

In [4]:
tr = td[td.game_date < HOLDOUT_START].copy()
ho = td[td.game_date >= HOLDOUT_START].copy()

mc_ho = MulticlassOptimalBinning('cornerRatio', solver='cp')
mc_ho.fit(tr.rawRatio.values, tr.threesMade.clip(upper=BIN_CAP).astype(int).values)
tr['cornerRatio'] = mc_ho.transform(tr.rawRatio.values)
ho['cornerRatio'] = mc_ho.transform(ho.rawRatio.values)

for col in [c for c in FEATS if c not in NOSCALE]:
    center, sd = tr[col].mean(), tr[col].std(ddof=0)
    tr[col] = (tr[col] - center) / sd
    ho[col] = (ho[col] - center) / sd

ho_fit = sm.MNLogit(tr.y.values, sm.add_constant(tr[FEATS])).fit(method='bfgs', maxiter=5000, disp=0)
p_ho = ho_fit.predict(sm.add_constant(ho[FEATS], has_constant='add')).to_numpy()

Y = np.eye(Y_MAX + 1)[ho.y.values]
null = np.bincount(tr.y.values, minlength=Y_MAX + 1) / len(tr)
model_brier = ((p_ho - Y) ** 2).sum(axis=1).mean()
null_brier = ((null - Y) ** 2).sum(axis=1).mean()
print('holdout rows {:,} | Brier {:.5f} | null {:.5f} | skill {:.4f}'.format(len(ho), model_brier, null_brier, 1 - model_brier / null_brier))

holdout rows 18,828 | Brier 0.71014 | null 0.77240 | skill 0.0806


## Final fit on all four seasons

In [5]:
fit_df = td.copy()

corner_bin = MulticlassOptimalBinning('cornerRatio', solver='cp')
corner_bin.fit(fit_df.rawRatio.values, fit_df.threesMade.clip(upper=BIN_CAP).astype(int).values)
fit_df['cornerRatio'] = corner_bin.transform(fit_df.rawRatio.values)
print('cornerRatio splits:', corner_bin.splits.round(3).tolist())

scaler = {}
for col in [c for c in FEATS if c not in NOSCALE]:
    scaler[col] = {'center': fit_df[col].mean(), 'var': fit_df[col].std(ddof=0), 'type': 'standard'}
    fit_df[col] = (fit_df[col] - scaler[col]['center']) / scaler[col]['var']

model = sm.MNLogit(fit_df.y.values, sm.add_constant(fit_df[FEATS])).fit(method='bfgs', maxiter=5000, disp=0)
print('converged: {} | rows {:,} | classes {}'.format(model.mle_retvals['converged'], int(model.nobs), Y_MAX + 1))
model.params.round(3)

cornerRatio splits: [0.525, 0.712, 0.785, 0.87]


converged: True | rows 72,886 | classes 10


,0,1,2,3,4,5,6,7,8
const,0.098,-0.302,-0.918,-1.763,-2.616,-3.567,-4.527,-5.562,-6.823
cornerRatio,-0.249,-0.339,-0.563,-0.565,-1.038,-0.997,-1.096,-0.702,-0.756
abv_interaction,-0.014,0.012,0.217,0.327,0.014,0.380,-0.057,0.633,0.562
crn_interaction,0.086,0.068,0.147,0.342,0.286,0.276,0.498,0.618,0.341
mvAvgThrees,0.127,0.241,0.302,0.425,0.343,0.403,0.545,0.602,0.835
daysBetweenGames,-0.021,-0.027,-0.050,-0.060,-0.068,-0.108,-0.109,-0.042,-0.148
wide_interaction,0.023,0.045,0.019,-0.067,-0.017,-0.040,-0.186,-0.407,-0.360
home,0.053,0.095,0.115,0.188,0.159,0.112,0.023,-0.002,0.508
crn_fgaMv,0.222,0.341,0.415,0.262,0.427,0.458,0.223,-0.022,0.271
abv_fgaMv,0.493,0.746,0.762,0.844,1.366,1.095,1.547,0.909,1.058


## Save artifacts
The `var` key holds the standard deviation, matching `standRobust_scaler` and the 2025-26 file.

In [6]:
os.makedirs(RUN_DIR, exist_ok=True)
pickle.dump(model, open(os.path.join(RUN_DIR, 'threeModel.pkl'), 'wb'))
pickle.dump(scaler, open(os.path.join(RUN_DIR, 'scaler.pkl'), 'wb'))
pickle.dump(corner_bin, open(os.path.join(RUN_DIR, 'cornerBin.pkl'), 'wb'))
sorted(os.listdir(RUN_DIR))

['cornerBin.pkl', 'scaler.pkl', 'threeModel.pkl']